## Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


# Haversine
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


# Нормализация колонок
def normalize_columns(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    return df


# Подготовка признаков (улучшено)
def prepare_mountain_features(df):
    df = normalize_columns(df)

    # автоопределение колонок
    date_col = [c for c in df.columns if 'date' in c or 'дата' in c][0]
    temp_col = [c for c in df.columns if 'temp' in c or 'темп' in c][0]
    precip_col = [c for c in df.columns if 'precip' in c or 'осадки' in c][0]

    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)

    # осадки
    df['p7'] = df[precip_col].rolling(7).sum()

    # положительные температуры
    df['t_pos'] = df[temp_col].clip(lower=0)
    df['degree_days'] = df['t_pos'].rolling(10).sum()

    # снег
    snow_cols = [c for c in df.columns if 'снег' in c or 'snow' in c]
    if snow_cols:
        df['snow_smooth'] = df[snow_cols[0]].rolling(5).mean()

    return df, date_col


# Лаговая корреляция
def best_lag_corr(series1, series2, max_lag=14):
    best_corr = 0
    best_lag = 0

    for lag in range(max_lag + 1):
        corr = series1.corr(series2.shift(lag))
        if pd.notna(corr) and abs(corr) > abs(best_corr):
            best_corr = corr
            best_lag = lag

    return best_corr, best_lag


# Оценка станции (улучшено)
def score_station(hydro_df, meteo_df, max_lag=14):
    hydro = normalize_columns(hydro_df)

    # автоопределение колонок
    date_col = [c for c in hydro.columns if 'дата' in c or 'date' in c][0]
    q_col = [c for c in hydro.columns if 'расход' in c or 'level' in c][0]

    hydro[date_col] = pd.to_datetime(hydro[date_col])
    hydro = hydro.set_index(date_col)[q_col]

    meteo_df, m_date_col = prepare_mountain_features(meteo_df)
    meteo_df = meteo_df.set_index(m_date_col)

    features = {
        'p7': 0.4,
        'degree_days': 0.3,
        'snow_smooth': 0.3
    }

    total_score = 0
    details = {}

    for feature, weight in features.items():
        if feature not in meteo_df.columns:
            continue

        combined = pd.concat([hydro, meteo_df[feature]], axis=1).dropna()
        combined.columns = ['Q', 'X']

        if len(combined) < 30:
            continue

        corr, lag = best_lag_corr(combined['Q'], combined['X'], max_lag)

        total_score += weight * abs(corr)
        details[feature] = {'corr': corr, 'lag': lag}

    return total_score, details


# Основная функция (топ версия)
def select_stations(hydro_df, meteo_folder, hydro_coords, coords_df, max_lag=15, L=100):

    results = []
    meteo_folder = Path(meteo_folder)
    coords_df = normalize_columns(coords_df)

    for file in meteo_folder.glob('*.xlsx'):
        name = file.stem

        row = coords_df[coords_df.iloc[:,0] == name]
        if row.empty:
            print(f'Нет координат для {name}')
            continue

        lat_station = row.iloc[0]['широта']
        lon_station = row.iloc[0]['долгота']

        meteo_df = pd.read_excel(file)

        score, details = score_station(hydro_df, meteo_df, max_lag)

        # расстояние
        dist = haversine(hydro_coords[0], hydro_coords[1], lat_station, lon_station)

        # экспоненциальное затухание (лучше физически)
        dist_weight = np.exp(-dist / L)

        final_score = score * dist_weight

        results.append({
            'station': name,
            'score': final_score,
            'distance_km': dist,
            'p7_corr': details.get('p7', {}).get('corr'),
            'degree_days_corr': details.get('degree_days', {}).get('corr'),
            'snow_corr': details.get('snow_smooth', {}).get('corr'),
        })

    df = pd.DataFrame(results)
    return df.sort_values('score', ascending=False)

In [2]:
def get_hydro_coords(id_post):
    df = pd.read_excel(
        "https://raw.githubusercontent.com/JujikMilkivey/datasets_for_forecasting/main/Data-structured/hydro_stations_coords.xlsx"
    )
    filtered = df[df['Код_поста'] == id_post]
    if filtered.empty:
        return None  # поста нет
    return filtered[['Широта', 'Долгота']].iloc[0]


hydro_coords = pd.read_excel("https://raw.githubusercontent.com/JujikMilkivey/datasets_for_forecasting/main/Data-structured/hydro_stations_coords.xlsx")
hydro_coords

,Код_поста,Река,Створ,Широта,Долгота,Бассейновый_округ,Пост_открыт,"Площадь_водосбора, км2","Расстояние от устья, км","Расстояние от истока, км","Ноль поста, м","Средняя высота водосбора, м","Средний уклон водотока, ‰","Озерность водосбора, %","Заболоченность водосбора, %","Залесенность водосбора, %"
0,84108,Терек,Владикавказ,43.0300,44.6800,Западно-Каспийский,31.07.1911 г.,1490.0,530.0,93.0,678.64 БС,2540.0,26.0,0,0,10
1,84119,Терек,ст-ца Котляревская,43.5800,44.0800,Западно-Каспийский,01.04.1902 г.,8920.0,437.0,186.0,212.82 БС,1800.0,16.0,0,0,20
2,84157,Цея,пгт Бурон,42.7820,43.9940,NaN,01.08.1950 г.,100.0,0.3,13.1,1203.01 БС,2820.0,120.0,0,0,15
3,84165,Гизельдон,с.Даргавс,42.8330,44.4330,NaN,01.10.1930 г.,129.0,66.0,14.0,1407.25 БС,2670.0,110.0,0,0,5
4,84192,Малка,с.Каменномостское,43.7300,43.0700,NaN,02.02.1926 г.,1540.0,135.0,75.0,792.72 БС,2000.0,32.0,0,0,10
5,84200,Малка,ст.Прохладная,43.7330,44.0760,NaN,28.04.1902 г.,9820.0,24.0,186.0,181.21 БС,1900.0,16.0,0,0,15
6,84213,Баксан,г.Тырныауз,43.2100,42.5600,NaN,01.01.1964 г.,838.0,127.0,42.0,1281.25 БС,3040.0,32.0,1,0,10
7,84215,Баксан,с.Заюково,43.6000,43.3000,NaN,01.02.1931 г.,2100.0,82.0,87.0,658.31 БС,2360.0,23.0,0,0,5
8,84233,Чегем 1-й,с.Нижний Чегем,43.5000,43.2800,NaN,16.03.1925 г.,739.0,48.0,55.0,875.76 БС,2500.0,41.0,0,0,10
9,84243,Черек Балкарский,с.Бабугент,43.2990,43.5860,NaN,13.02.1930 г.,695.0,3.0,51.0,746.53 БС,2590.0,36.0,0,0,5


In [3]:
id_post = 84108
df = select_stations(
    hydro_df=pd.read_excel(f"https://raw.githubusercontent.com/JujikMilkivey/datasets_for_forecasting/main/Data-structured/%D0%97%D0%B0%D0%BF%D0%B0%D0%B4%D0%BD%D0%BE-%D0%9A%D0%B0%D1%81%D0%BF%D0%B8%D0%B9%D1%81%D0%BA%D0%B8%D0%B9/{id_post}.xlsx"),
    meteo_folder=r"C:\Users\UserHome\Desktop\Аспирантура\Forecasting_2025\datasets_for_forecasting\Data-structured\Meteo",
    hydro_coords=list(get_hydro_coords(id_post)),
    coords_df=pd.read_excel("https://raw.githubusercontent.com/JujikMilkivey/datasets_for_forecasting/main/Data-structured/meteo_stations_coords.xlsx")
)
df

,station,score,distance_km,p7_corr,degree_days_corr,snow_corr
17,Vladikavkaz,0.535295,3.711058,0.452040,0.822269,-0.426791
6,Grozniy,0.172126,90.022081,0.216330,0.822647,-0.300429
15,Sulak_visokogornaya,0.128645,146.146767,0.365670,0.676136,-0.685479
7,Gudermes,0.119279,122.042437,0.172115,0.822926,-0.294891
13,Schadjatmaz,0.107303,180.122717,0.551430,0.760051,-0.671191
8,Kislovodsk,0.089759,185.410105,0.495160,0.817807,-0.432635
12,Min_Vody,0.078721,185.990878,0.380866,0.823390,-0.354251
18,Yuzhno-Sukhokumsk,0.073544,173.469483,0.196783,0.835837,-0.291077
3,Budennovsk,0.066927,202.222446,0.380866,0.823390,-0.354251
4,Buynaksk,0.058016,200.029999,0.226612,0.823028,-0.304205
